# Generate part02 data for 2021 JLab Hackathon

This notebook is used to generate the data sets for part02 of the 2021 Hackathon.

This problem is identical in setup to the part01 problem in that is has a 30x30 block calorimeter with a single shower. For this though, each image has a partner image with noise added. The participants should implement an autoencoder (or other AI/ML solution) that produces the denoisified image from the noisy one.

Similar to part01, the "judge" data sets will have the clean images stored only in the "restricted" directory.

n.b. It is tempting to just use the images from part01 here, but because that would give participants access to the clean "judge" images, we need to regenerate those anyway. Thus, we just regenerate all data sets.

This first section defines several gobal parameters defining how to create the data set.

In [1]:
import math

#------------------------------------------
# Configuration
Nblocks  = 30
Nevents_train   = 10000
Nevents_test    =  2000
Nevents_judge   =  2000
Nclusters = 1

Xmin            = -1000.0
Xmax            = +1000.0
ClusterSize     = 1.0  # cluster std dev in blockwidths
ClusterAmpMean  = 3.0  # Mean amplitude of cluster 
ClusterAmpSigma = 2.0  # Standard deviation of cluster amplitude
ClusterAmpMin   = 0.5  # Minimum cluster amplitude
ClusterAmpMax   = 8.0  # Maximum cluster amplitude
ClusterPosSigma = 8.0  # Standard deviation of Gaussian used to sample position of cluster

NoiseHit_probability = 0.3  # fraction of blocks that will have a noise hit
NoiseHit_AmpMean     = ClusterAmpMean/30.0    # Mean size of noise hit
NoiseHit_AmpSigma    = NoiseHit_AmpMean       # sigma for noise hit size
NoiseHit_AmpMin      = ClusterAmpMin/10.0     # min size for a noise hit
NoiseHit_AmpMax      = 20.0 * NoiseHit_AmpMin # max size for a noise hit

#------------------------------------------
# Calculated values
binwidth = (Xmax-Xmin)/Nblocks
one_over_sqrt2_sigma = 1.0 / (math.sqrt(2) * ClusterSize*binwidth)
int_gauss = 2.0*math.erf(binwidth/2.0*one_over_sqrt2_sigma)
max_integral = ClusterAmpMax*int_gauss*int_gauss  # maximum value a block can have


The following is just a utility to print an updating progress bar so we can monitor the longer running cells.

In [2]:
#------------------------------------------
# Code for displaying progess bar (ripped from internet: https://www.mikulskibartosz.name/how-to-display-a-progress-bar-in-jupyter-notebook/)
import time, sys
from IPython.display import clear_output

def update_progress(label, progress):
    bar_length = 20
    if isinstance(progress, int):
        progress = float(progress)
    if not isinstance(progress, float):
        progress = 0
    if progress < 0:
        progress = 0
    if progress >= 1:
        progress = 1

    block = int(round(bar_length * progress))

    clear_output(wait = True)
    text = label + ": [{0}] {1:.1f}%".format( "#" * block + "-" * (bar_length - block), progress * 100)
    print(text)

## Procedures to create single events and full event sets in CSV format

This is where the events are generated and the block values defined. The routine
is used to create training, test, and judging data sets. 
n.b. The judging sets will have the labels separated manually in a later cell. 

Here, we create "pure" events. A cluster is defined by a point randomly selected
to be in the detector. The centroid may be at the border, but not beyond it.
The distribution in x/y is sampled from a Gaussian that is centered on the center
of the detector (0,0) with a sigma that is ClusterPosSigma blocks. This is done to
give the position a non-uniform distribution, but still be wide enough to ensure
events cover the entire detector.

The amplitude of the cluster is randomly selected, also from a Gaussian distribution,
but with limits defined by ClusterAmpMin and ClusterAmpMax. 

Each cell of the NblocksxNblocks array is set equal to the integral of a 2D
Gaussian centered on the cluster, but with a sigma of ClusterSize\*binwidth. The
Gaussian itself is not properly normalized, but this is not really needed since the
"energy" units of the clusters are arbitrary.

It turns out that the integral of a 2D Guassian over a square block is proportional
to the product of 2 erf differences. We only need to calculate the erf(x2)-erf(x1)
for each column(row) and the cell is the product of these.

In [3]:
import numpy as np
import math
import os

#------------------------------------------
# MakeEvent
#------------------------------------------
def MakeEvent():

    one_over_sqrt2_sigma = 1.0 / (math.sqrt(2) * ClusterSize*binwidth)

    for icluster in range(Nclusters):
        
        # Sample amplitude of cluster
        amp = 0.0
        while amp<ClusterAmpMin or amp>ClusterAmpMax : amp = np.random.normal( ClusterAmpMean, ClusterAmpSigma )
        
        # Sample X/Y coordinate of cluster
        x0 = -10000.0
        y0 = -10000.0
        while x0<Xmin or x0>Xmax : x0 = np.random.normal( 0.0, ClusterPosSigma*binwidth )
        while y0<Xmin or y0>Xmax : y0 = np.random.normal( 0.0, ClusterPosSigma*binwidth )
        
        # Calculate factors for integrals over blocks
        xfac = [0]*(Nblocks+1)
        yfac = [0]*(Nblocks+1)
        for i in range(Nblocks+1):
            x = Xmin + i*binwidth
            a0 = (x-x0)*one_over_sqrt2_sigma
            a1 = (x+binwidth-x0)*one_over_sqrt2_sigma
            xfac[i] = math.erf( a1 ) - math.erf( a0 )
        for i in range(Nblocks+1):
            y = Xmin + i*binwidth
            a0 = (y-y0)*one_over_sqrt2_sigma
            a1 = (y+binwidth-y0)*one_over_sqrt2_sigma
            yfac[i] = math.erf( a1 ) - math.erf( a0 )
       
        # Set all block values
        Eblock = [0.0] * Nblocks * Nblocks
        for i in range(Nblocks):
            for j in range(Nblocks):
                idx = i + Nblocks*j
                Eblock[idx] = '%6.4f' % (amp*xfac[i]*yfac[j])
                    
        # Create CSV output string for this event
        event = ','.join(Eblock) + ',' + ','.join(['%6.4f' % x for x in [amp, x0, y0]])

        return event

#------------------------------------------
# MakeDataSet
#------------------------------------------
def MakeDataSet(directory, setname, Nevents):

    # Create directory (if needed) and open CSV file
    os.makedirs(directory, exist_ok=True)
    of = open( os.path.join(directory, setname + '.csv'), 'w')
    label = 'Generating ' + os.path.join(directory, setname)

    for ievent in range(Nevents):
        event_csv = MakeEvent() # Make one event w/ labels
        of.write( event_csv + '\n')
        if ievent%100 == 0 : update_progress(label, ievent / Nevents) # Periodically update progress bar
    of.close() # Close output CSV file    
    update_progress(label, 1)
    print('Done.')

## Generate all part02 data sets

In [4]:
MakeDataSet('part02', 'train', Nevents_train)

Generating part02/train: [####################] 100.0%
Done.


In [5]:
MakeDataSet('part02', 'test' , Nevents_test)

Generating part02/test: [####################] 100.0%
Done.


In [6]:
MakeDataSet('part02', 'judge', Nevents_judge)

Generating part02/judge: [####################] 100.0%
Done.


## Generate clean and noisy image files from CSV

Here the CSV file with "clean" events is read and a PNG is produced for each event and written to a dedicated images directory. The noisy images are also produced here. A new CSV file is also created in the output directory which has the noisy and clean image file names as the only two values for each event.

We do not provide a CSV formatted file with the values for the noisy images. 

In [7]:
import csv
from PIL import Image
import matplotlib.pyplot as plt
import os

def GenerateImages(directory, setname):

    image_dirname = os.path.join(directory, setname+'_images')
    os.makedirs(image_dirname, exist_ok=True)
    label = 'Generating PNG images ' + os.path.join(directory, setname)

    with open( os.path.join(directory, setname + '.csv'), newline='' ) as csvfile:
        
        Nevents = sum(1 for line in csvfile)
        csvfile.seek(0) # rewind
        
        newcsv_file = open( os.path.join(directory, setname + '_images.csv'), 'w' )
      
        reader = csv.reader(csvfile)
        for i, line in enumerate(reader):

            # Periodically update progress bar
            if i%100 == 0 : update_progress(label, i / Nevents)

            # Write clean image file
            X = np.array([255.0*float(x)/max_integral for x in line[:Nblocks*Nblocks]]).reshape(Nblocks,Nblocks).astype(np.uint8)
            img = Image.fromarray(X)
            clean_fname = os.path.join(image_dirname, 'clean_event%06d.png' % i)
            img.save(clean_fname)
            
            # Add noise
            NnoiseHits = np.random.binomial(Nblocks*Nblocks, NoiseHit_probability)       # Select number of noise hits from binomial distribution
            hitAmps = np.random.normal(NoiseHit_AmpMean, NoiseHit_AmpSigma, NnoiseHits)  # Randomly sample amplitude of NnoiseHits
            np.clip(hitAmps, NoiseHit_AmpMin, NoiseHit_AmpMax, out=hitAmps)              # Force hits within limits
            noise = np.zeros(Nblocks*Nblocks)                                            # Initialize noise to all zeros
            noise[:NnoiseHits] = hitAmps                                                 # Replace first NnoiseHits elements of noise array with noise hit values
            np.random.shuffle(noise)                                             # Randomly shuffle values to distribute noise hits throughout array
            
            Y = noise + np.array([float(x) for x in line[:Nblocks*Nblocks]])
            Y = np.clip( 255.0*Y/max_integral, 0.0, 255.0 ).reshape(Nblocks,Nblocks).astype(np.uint8)            
            
            # Write noisy image file
            img = Image.fromarray(Y)
            noisy_fname = os.path.join(image_dirname, 'event%06d.png' % i)
            img.save(noisy_fname)

            rel_clean_fname = '/'.join(clean_fname.split('/')[1:]) # drop first directory from path (i.e. 'part02/') so CSV only contains setname/filename.png
            rel_noisy_fname = '/'.join(noisy_fname.split('/')[1:]) # drop first directory from path (i.e. 'part02/') so CSV only contains setname/filename.png
            newcsv_line = ', '.join([noisy_fname, clean_fname])
            newcsv_file.write(newcsv_line + '\n')
 
        newcsv_file.close()

    # Final update of progress bar
    update_progress(label, 1)
    print('Done.')

Matplotlib created a temporary config/cache directory at /tmp/matplotlib-tmqv66o8 because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [8]:
 GenerateImages('part02', 'train')

Generating PNG images part02/train: [####################] 100.0%
Done.


In [9]:
 GenerateImages('part02', 'test')

Generating PNG images part02/test: [####################] 100.0%
Done.


In [10]:
GenerateImages('part02', 'judge')

Generating PNG images part02/judge: [####################] 100.0%
Done.


## Display some images

In [2]:
from IPython.display import HTML

Npics_per_row = 4
pic_width="300px"

mess = '<h3>NOTE: Images may appear correctly due to scaling algorithm of browser!</h3><br>'

#---------------------
# Train
mess += '<table><tr>'
mess += '<td colspan="100"> <center>TRAIN</center> </td>\n'
mess += '</tr><tr>\n'
for i in range(2543, 2543+Npics_per_row):
    noisy_fname = 'part02/train_images/event%06d.png' % i
    clean_fname = 'part02/train_images/clean_event%06d.png' % i
    mess += '<td>\n'
    mess += '<img width="' + pic_width + '" src="' + clean_fname + '"><br>\n'
    mess += '<img width="' + pic_width + '" src="' + noisy_fname + '"><br>\n'
    mess += '<font size="-1">' + noisy_fname + '</font>\n'
    mess += '</td>\n'
mess += '</tr></table>'
display(HTML(mess))

#---------------------
# Test
mess = '<table><tr>'
mess += '<td colspan="100"> <center>TEST</center> </td>\n'
mess += '</tr><tr>\n'
for i in range(Npics_per_row):
    noisy_fname = 'part02/test_images/event%06d.png' % i
    clean_fname = 'part02/test_images/clean_event%06d.png' % i
    mess += '<td>\n'
    mess += '<img width="' + pic_width + '" src="' + clean_fname + '"><br>\n'
    mess += '<img width="' + pic_width + '" src="' + noisy_fname + '"><br>\n'
    mess += '<font size="-1">' + noisy_fname + '</font>\n'
    mess += '</td>\n'
mess += '</tr></table>'
display(HTML(mess))

#---------------------
# Judge
mess = '<table><tr>'
mess += '<td colspan="100"> <center>JUDGE</center> </td>\n'
mess += '</tr><tr>\n'
for i in range(Npics_per_row):
    noisy_fname = 'part02/judge_images/event%06d.png' % i
    clean_fname = 'part02/judge_images/clean_event%06d.png' % i
    mess += '<td>\n'
    mess += '<img width="' + pic_width + '" src="' + clean_fname + '"><br>\n'
    mess += '<img width="' + pic_width + '" src="' + noisy_fname + '"><br>\n'
    mess += '<font size="-1">' + noisy_fname + '</font>\n'
    mess += '</td>\n'
mess += '</tr></table>'
display(HTML(mess))


## Scrub Judging Files

In this cell the csv file with the clean data for the "judge" data set as well as the clean image PNG files are moved to the directory "restricted". 


In [12]:
import os
import shutil
import csv
from glob import *

# Make sure "restricted" directory exists and move files with labels there
os.makedirs('restricted/part02/judge_images', exist_ok=True)
if os.path.exists('part02/judge.csv'):
    shutil.move('part02/judge.csv', 'restricted/part02_judge.csv')

# Move all "clean" judge PNG files to the restricted area
files = glob('part02/judge_images/clean_*.png')
for f in files:
    newfname = f.replace('part02/judge_images', 'restricted/part02/judge_images')
    os.rename(f, newfname)

             